# LFP and Motion Energy


In [1]:
import sys
import importlib
from pathlib import Path

import matplotlib

try:
    ip = get_ipython()
except NameError:
    ip = None

if ip is not None:
    try:
        ip.run_line_magic("matplotlib", "qt")
    except Exception as exc:
        print(f"Could not enable %matplotlib qt, using {matplotlib.get_backend()}: {exc}")
else:
    try:
        matplotlib.use("QtAgg", force=True)
    except Exception as exc:
        print(f"Could not enable QtAgg backend, using {matplotlib.get_backend()}: {exc}")

import matplotlib.pyplot as plt
import numpy as np

import lfp_motion_energy_utils as lme_utils
lme_utils = importlib.reload(lme_utils)
from lfp_motion_energy_utils import (
    amplitude_envelope_1d,
    CameraSyncSettings,
    benchmark_lfp_loading,
    build_lfp_motion_energy_cache,
    butter_filter_1d,
    channel_range,
    clean_motion_energy_regular_artifacts,
    compute_lfp_magnitude_and_me,
    compute_lfp_magnitude_and_me_from_cache,
    correlate_unit_rates_to_trace,
    decimate_spike_rate_result,
    decimate_for_lag,
    default_lfp_motion_energy_cache_path,
    future_trace_change,
    gaussian_smooth_rows,
    lagged_pearson,
    load_channel_average_lfp,
    load_kilosort_spikes,
    load_lfp_motion_energy_cache,
    load_motion_energy,
    load_recording,
    map_spike_rates_to_timebase,
    pearsonr_valid,
    percentile_minmax,
    print_recording_summary,
    subtract_static_channel_mean,
    unit_trace_lag_correlations,
    zscore,
)


In [2]:
# Dataset
EXPERIMENT_PATH = Path("/home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1")


In [3]:
# Helper functions
def corr_filter_settings() -> dict:
    corr_stop = recording.lfp.n_lfp_channels if CORR_CHANNEL_STOP is None else CORR_CHANNEL_STOP
    return {
        "channels": tuple(int(ch) for ch in channel_range(CORR_CHANNEL_START, corr_stop, CORR_CHANNEL_STEP)),
        "start_s": float(CORR_START_S),
        "duration_s": None if CORR_DURATION_S is None else float(CORR_DURATION_S),
        "chunk_seconds": float(CORR_CHUNK_SECONDS),
        "smooth_seconds": float(CORR_SMOOTH_SECONDS),
        "target_rate_hz": float(TARGET_RATE_HZ),
        "read_chunk_seconds": float(LFP_READ_CHUNK_SECONDS),
        "read_mode": str(LFP_READ_MODE),
        "bandpass_hz": None if CORR_LFP_BANDPASS_HZ is None else tuple(float(v) for v in CORR_LFP_BANDPASS_HZ),
    }


def update_corr_trace_if_needed(force: bool = False):
    global corr_channels, corr_trace, corr_r, corr_valid, corr_lfp_label, corr_me_label, corr_trace_settings
    settings = corr_filter_settings()
    if force or globals().get("corr_trace_settings") != settings:
        corr_channels = np.asarray(settings["channels"], dtype=np.int64)
        corr_trace = lme_utils.compute_lfp_magnitude_and_me(
            recording,
            corr_channels,
            start_s=settings["start_s"],
            duration_s=settings["duration_s"],
            chunk_seconds=settings["chunk_seconds"],
            smooth_seconds=settings["smooth_seconds"],
            target_rate_hz=settings["target_rate_hz"],
            read_chunk_seconds=settings["read_chunk_seconds"],
            read_mode=settings["read_mode"],
            lfp_bandpass_hz=settings["bandpass_hz"],
        )
        corr_r, corr_valid = pearsonr_valid(corr_trace.lfp_trace, corr_trace.motion_energy)
        if settings["bandpass_hz"] is None:
            corr_lfp_label = "mean LFP envelope, z"
            corr_me_label = "motion-energy envelope, z"
        else:
            low_hz, high_hz = settings["bandpass_hz"]
            corr_lfp_label = f"mean LFP envelope {low_hz:g}-{high_hz:g} Hz, z"
            corr_me_label = f"motion-energy envelope {low_hz:g}-{high_hz:g} Hz, z"
        corr_trace_settings = settings
        print(
            f"Pearson r = {corr_r:.4f} over {corr_valid.sum():,} valid samples "
            f"at {corr_trace.sample_rate_hz:.1f} Hz"
        )
        print(
            "Envelope minima before z-scoring: "
            f"LFP={np.nanmin(corr_trace.lfp_trace):.4g}, "
            f"ME={np.nanmin(corr_trace.motion_energy):.4g}"
        )
    return corr_trace


In [4]:
# Load recording metadata, camera TTLs, and motion energy
CAMERA = CameraSyncSettings(
    channel=1,
    threshold_mv=3000.0,
    min_separation_samples=85,
    force_redetect=False,
)

ME_ARTIFACT_FILTER = False
ME_ARTIFACT_METHOD = "fixed"       # "fixed" or "peak_regular"
ME_ARTIFACT_DIFF_HEIGHT = None
ME_ARTIFACT_PERIOD_FRAMES = 250
ME_ARTIFACT_BEFORE_FRAMES = 3
ME_ARTIFACT_AFTER_FRAMES = 10

recording = load_recording(EXPERIMENT_PATH, camera=CAMERA)
if ME_ARTIFACT_FILTER:
    artifact_frames = recording.apply_motion_energy_artifact_filter(
        method=ME_ARTIFACT_METHOD,
        diff_height=ME_ARTIFACT_DIFF_HEIGHT,
        period_frames=ME_ARTIFACT_PERIOD_FRAMES,
        before_frames=ME_ARTIFACT_BEFORE_FRAMES,
        after_frames=ME_ARTIFACT_AFTER_FRAMES,
    )
    print(f"removed/interpolated {len(artifact_frames):,} regular motion-energy artifact frames")
print_recording_summary(recording)


nChan: 385, nFileSamp: 19877151
loaded 591,868 cached camera TTLs from /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/CAM_TTLs.npy
nidq: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/20260714_ant_g0/20260714_ant_g0_t0.nidq.bin
lfp:  /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/20260714_ant_g0/20260714_ant_g0_imec0/20260714_ant_g0_t0.imec0.lf.bin
me:   /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/cam1_2026-07-14-11-05-52.avi.me
nidq rate=10593.400 Hz; lfp rate=2500.000 Hz
frames=591,868; video duration=132.31 min
lfp channels=384; lfp duration=132.51 min


In [ ]:
# Build/load reusable 250 Hz LFP/ME cache
LFP_CACHE_BUILD = False
LFP_CACHE_LOAD = False
LFP_CACHE_OVERWRITE = False
LFP_CACHE_TARGET_RATE_HZ = 250.0
LFP_CACHE_CHANNEL_START = 0
LFP_CACHE_CHANNEL_STOP = None
LFP_CACHE_CHANNEL_STEP = 1
LFP_CACHE_START_S = 0.0
LFP_CACHE_DURATION_S = None
LFP_CACHE_CHUNK_SECONDS = 60.0
LFP_CACHE_PATH = None
LFP_READ_CHUNK_SECONDS = 1.0
LFP_READ_MODE = "auto"

lfp_cache_stop = recording.lfp.n_lfp_channels if LFP_CACHE_CHANNEL_STOP is None else LFP_CACHE_CHANNEL_STOP
lfp_cache_channels = channel_range(
    LFP_CACHE_CHANNEL_START,
    lfp_cache_stop,
    LFP_CACHE_CHANNEL_STEP,
)
lfp_cache_path = (
    default_lfp_motion_energy_cache_path(
        recording,
        lfp_cache_channels,
        target_rate_hz=LFP_CACHE_TARGET_RATE_HZ,
    )
    if LFP_CACHE_PATH is None
    else Path(LFP_CACHE_PATH)
)

if LFP_CACHE_BUILD:
    lfp_cache_path = build_lfp_motion_energy_cache(
        recording,
        lfp_cache_path,
        lfp_cache_channels,
        target_rate_hz=LFP_CACHE_TARGET_RATE_HZ,
        start_s=LFP_CACHE_START_S,
        duration_s=LFP_CACHE_DURATION_S,
        chunk_seconds=LFP_CACHE_CHUNK_SECONDS,
        read_chunk_seconds=LFP_READ_CHUNK_SECONDS,
        read_mode=LFP_READ_MODE,
        overwrite=LFP_CACHE_OVERWRITE,
    )
elif lfp_cache_path.exists():
    print(f"LFP/ME cache exists: {lfp_cache_path}")
else:
    print(f"No LFP/ME cache yet: {lfp_cache_path}")
    print("Set LFP_CACHE_BUILD=True and rerun this cell to create it.")

lfp_me_cache = None
if LFP_CACHE_LOAD and lfp_cache_path.exists():
    lfp_me_cache = load_lfp_motion_energy_cache(
        lfp_cache_path,
        start_s=LFP_CACHE_START_S,
        duration_s=LFP_CACHE_DURATION_S,
        channels=lfp_cache_channels,
    )
    print(
        f"loaded raw cache window: {len(lfp_me_cache.channels)} channel(s), "
        f"{len(lfp_me_cache.t_video_s):,} samples at {lfp_me_cache.sample_rate_hz:.1f} Hz"
    )


In [ ]:
# Tune motion-energy artifact filter
TUNE_ME_START_FRAME = 0
TUNE_ME_N_FRAMES = 5000
TUNE_ME_METHOD = ME_ARTIFACT_METHOD
TUNE_ME_DIFF_HEIGHT = ME_ARTIFACT_DIFF_HEIGHT
TUNE_ME_PERIOD_FRAMES = ME_ARTIFACT_PERIOD_FRAMES
TUNE_ME_BEFORE_FRAMES = ME_ARTIFACT_BEFORE_FRAMES
TUNE_ME_AFTER_FRAMES = ME_ARTIFACT_AFTER_FRAMES

_me_path, tune_me_raw, _me_attrs = load_motion_energy(EXPERIMENT_PATH)
tune_me_clean, tune_artifact_frames = clean_motion_energy_regular_artifacts(
    tune_me_raw,
    method=TUNE_ME_METHOD,
    diff_height=TUNE_ME_DIFF_HEIGHT,
    period_frames=TUNE_ME_PERIOD_FRAMES,
    before_frames=TUNE_ME_BEFORE_FRAMES,
    after_frames=TUNE_ME_AFTER_FRAMES,
)

tune_stop_frame = min(len(tune_me_raw), TUNE_ME_START_FRAME + TUNE_ME_N_FRAMES)
tune_window = slice(TUNE_ME_START_FRAME, tune_stop_frame)
tune_x = np.arange(TUNE_ME_START_FRAME, tune_stop_frame)
tune_in_window = tune_artifact_frames[
    (tune_artifact_frames >= TUNE_ME_START_FRAME) & (tune_artifact_frames < tune_stop_frame)
]

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, constrained_layout=True)
axes[0].plot(tune_x, tune_me_raw[tune_window], lw=0.7, label="raw ME")
axes[0].plot(tune_x, tune_me_clean[tune_window], lw=0.9, label="cleaned ME")
if tune_in_window.size:
    axes[0].scatter(tune_in_window, tune_me_raw[tune_in_window], s=18, color="tab:red", label="artifact frames")
axes[0].set_ylabel("motion energy")
axes[0].set_title(
    f"ME artifact tuning: method={TUNE_ME_METHOD}, artifacts={len(tune_artifact_frames):,}, "
    f"window={TUNE_ME_BEFORE_FRAMES} before/{TUNE_ME_AFTER_FRAMES} after"
)
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(tune_x, tune_me_raw[tune_window] - tune_me_clean[tune_window], lw=0.7)
axes[1].axhline(0, color="0.5", lw=0.8)
axes[1].set_ylabel("raw - cleaned")
axes[1].grid(alpha=0.25)

tune_diff = np.diff(tune_me_raw.astype(np.float64, copy=False))
tune_diff_x = np.arange(TUNE_ME_START_FRAME, max(TUNE_ME_START_FRAME, tune_stop_frame - 1))
axes[2].plot(tune_diff_x, tune_diff[tune_window.start : max(tune_window.start, tune_window.stop - 1)], lw=0.7)
if TUNE_ME_METHOD == "peak_regular":
    tune_height = TUNE_ME_DIFF_HEIGHT
    if tune_height is None:
        finite_diff = tune_diff[np.isfinite(tune_diff)]
        tune_height = float(np.nanmedian(finite_diff) + 8.0 * np.nanstd(finite_diff))
    axes[2].axhline(tune_height, color="tab:red", lw=0.9, label=f"diff height={tune_height:.3g}")
    axes[2].legend()
axes[2].set_xlabel("frame")
axes[2].set_ylabel("diff(ME)")
axes[2].grid(alpha=0.25)
plt.show()

if tune_artifact_frames.size > 1:
    print(
        f"artifact frames: n={len(tune_artifact_frames):,}, "
        f"median spacing={np.median(np.diff(tune_artifact_frames)):.1f} frames, "
        f"first={tune_artifact_frames[:10]}"
    )
else:
    print(f"artifact frames: n={len(tune_artifact_frames):,}, first={tune_artifact_frames[:10]}")


In [5]:
# Load a chunk of LFP near target rate and plot as an image
LFP_IMAGE_START_S = 0.0
LFP_IMAGE_DURATION_S = 20.0
LFP_IMAGE_CHANNEL_START = 150
LFP_IMAGE_CHANNEL_STOP = 160
LFP_IMAGE_CHANNEL_STEP = 2
TARGET_RATE_HZ = 250.0
LFP_READ_CHUNK_SECONDS = 1.0
LFP_READ_MODE = "direct"
LFP_IMAGE_USE_CACHE = True
LFP_IMAGE_CLOSE_OLD_FIGURES = False

image_channels = channel_range(
    LFP_IMAGE_CHANNEL_START,
    LFP_IMAGE_CHANNEL_STOP,
    LFP_IMAGE_CHANNEL_STEP,
)
importlib.reload(lme_utils)

lfp_image_cache_path = None
if LFP_IMAGE_USE_CACHE:
    lfp_image_cache_path = lme_utils.find_lfp_motion_energy_cache(
        recording,
        image_channels,
        target_rate_hz=TARGET_RATE_HZ,
        start_s=LFP_IMAGE_START_S,
        stop_s=LFP_IMAGE_START_S + LFP_IMAGE_DURATION_S,
    )

if lfp_image_cache_path is not None:
    print(f"using cached LFP image data: {lfp_image_cache_path}")
    lfp_image = lme_utils.load_lfp_motion_energy_cache(
        lfp_image_cache_path,
        start_s=LFP_IMAGE_START_S,
        duration_s=LFP_IMAGE_DURATION_S,
        channels=image_channels,
    )
else:
    print("LFP image cache not found; loading directly from raw LF file")
    lfp_image = recording.load_lfp_downsampled(
        LFP_IMAGE_START_S,
        LFP_IMAGE_DURATION_S,
        image_channels,
        target_rate_hz=TARGET_RATE_HZ,
        read_chunk_seconds=LFP_READ_CHUNK_SECONDS,
        read_mode=LFP_READ_MODE,
    )
if lfp_image.t_video_s.size == 0:
    raise ValueError("No LFP samples in requested image window")

lfp_demeaned = subtract_static_channel_mean(lfp_image.lfp_uv)
print(
    f"loaded LFP image array: {lfp_demeaned.shape[0]} channel(s) x "
    f"{lfp_demeaned.shape[1]:,} sample(s), {lfp_demeaned.nbytes / 1024**2:.2f} MiB, "
    f"sample_rate={lfp_image.sample_rate_hz:.1f} Hz"
)

vlim = np.nanpercentile(np.abs(lfp_demeaned), 99)
if not np.isfinite(vlim) or vlim <= 0:
    vlim = 1.0
if LFP_IMAGE_CLOSE_OLD_FIGURES:
    plt.close("all")
plt.ion()
fig, ax = plt.subplots(figsize=(13, 7), constrained_layout=True)
im = ax.imshow(
    lfp_demeaned,
    aspect="auto",
    interpolation="nearest",
    cmap="RdBu_r",
    vmin=-vlim,
    vmax=vlim,
    extent=[lfp_image.t_video_s[0], lfp_image.t_video_s[-1], image_channels[-1], image_channels[0]],
)
ax.set_title(
    f"LFP at {lfp_image.sample_rate_hz:.1f} Hz, static channel-mean subtracted, "
    f"{LFP_IMAGE_START_S:.1f}-{LFP_IMAGE_START_S + LFP_IMAGE_DURATION_S:.1f} s"
)
ax.set_xlabel("video time (s)")
ax.set_ylabel("saved LF channel")
fig.colorbar(im, ax=ax, label="mean-subtracted LFP (uV)")
fig.canvas.draw_idle()
plt.show(block=False)


using cached LFP image data: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_cache_250hz_ch000-383.h5
loaded LFP image array: 5 channel(s) x 5,000 sample(s), 0.10 MiB, sample_rate=250.0 Hz


In [6]:
# Plot LFP magnitude and motion energy only, without Kilosort units
NO_UNIT_LFP_CHANNELS = list(range(0, 2))
NO_UNIT_PLOT_START_MIN = 0.0
NO_UNIT_PLOT_DURATION_MIN = None  # None plots all available LFP/ME samples after START_MIN
NO_UNIT_LFP_BANDPASS_HZ = (2.0, 50.0)
NO_UNIT_SMOOTH_SECONDS = 0.0
NO_UNIT_CACHE_CHUNK_SECONDS = 60.0

from time import perf_counter
import lfp_motion_energy_utils as lme_utils
lme_utils = importlib.reload(lme_utils)

no_unit_channels = np.asarray(NO_UNIT_LFP_CHANNELS, dtype=np.int64)
if no_unit_channels.size == 0:
    raise ValueError("NO_UNIT_LFP_CHANNELS must contain at least one LF channel")

no_unit_target_rate_hz = 250.0
no_unit_start_s = NO_UNIT_PLOT_START_MIN * 60.0
no_unit_duration_s = None if NO_UNIT_PLOT_DURATION_MIN is None else NO_UNIT_PLOT_DURATION_MIN * 60.0
no_unit_window_label = "full available window" if no_unit_duration_s is None else f"{NO_UNIT_PLOT_DURATION_MIN:g} min window"

overlap_start_s, overlap_stop_s = recording.lfp_video_overlap()
no_unit_request_start_s = max(no_unit_start_s, overlap_start_s)
no_unit_request_stop_s = (
    overlap_stop_s
    if no_unit_duration_s is None
    else min(no_unit_request_start_s + no_unit_duration_s, overlap_stop_s)
)
if no_unit_request_stop_s <= no_unit_request_start_s:
    raise ValueError("No LFP/video overlap in requested no-unit plot window")

no_unit_cache_path = lme_utils.find_lfp_motion_energy_cache(
    recording,
    no_unit_channels,
    target_rate_hz=no_unit_target_rate_hz,
    start_s=no_unit_request_start_s,
    stop_s=no_unit_request_stop_s,
)
if no_unit_cache_path is None:
    print("no-unit LFP/ME cache: not found, reading raw SpikeGLX LF chunks")
else:
    print(f"no-unit LFP/ME cache: {no_unit_cache_path}")

no_unit_trace_settings = {
    "channels": tuple(int(ch) for ch in no_unit_channels),
    "start_s": float(no_unit_start_s),
    "duration_s": None if no_unit_duration_s is None else float(no_unit_duration_s),
    "bandpass_hz": None if NO_UNIT_LFP_BANDPASS_HZ is None else tuple(float(v) for v in NO_UNIT_LFP_BANDPASS_HZ),
    "smooth_seconds": float(NO_UNIT_SMOOTH_SECONDS),
    "target_rate_hz": float(no_unit_target_rate_hz),
    "cache_chunk_seconds": float(NO_UNIT_CACHE_CHUNK_SECONDS),
}

no_unit_t0 = perf_counter()
if globals().get("_no_unit_trace_settings") == no_unit_trace_settings and "no_unit_trace" in globals():
    print("reusing existing no_unit_trace for unchanged settings")
elif no_unit_cache_path is not None:
    t_parts = []
    lfp_parts = []
    me_parts = []
    chunk_start_s = no_unit_request_start_s
    last_t = -np.inf
    chunk_i = 0
    while chunk_start_s < no_unit_request_stop_s:
        chunk_i += 1
        chunk_duration_s = min(NO_UNIT_CACHE_CHUNK_SECONDS, no_unit_request_stop_s - chunk_start_s)
        cache_chunk = lme_utils.load_lfp_motion_energy_cache(
            no_unit_cache_path,
            start_s=chunk_start_s,
            duration_s=chunk_duration_s,
            channels=no_unit_channels,
        )
        if cache_chunk.t_video_s.size:
            chunk_trace = lme_utils.compute_lfp_magnitude_and_me_from_cache(
                cache_chunk,
                smooth_seconds=NO_UNIT_SMOOTH_SECONDS,
                lfp_bandpass_hz=NO_UNIT_LFP_BANDPASS_HZ,
            )
            keep = chunk_trace.t_video_s > last_t
            if keep.any():
                t_parts.append(chunk_trace.t_video_s[keep])
                lfp_parts.append(chunk_trace.lfp_trace[keep])
                me_parts.append(chunk_trace.motion_energy[keep])
                last_t = float(chunk_trace.t_video_s[keep][-1])
        chunk_start_s += chunk_duration_s
        print(
            f"processed cached chunk {chunk_i} through "
            f"{min(chunk_start_s, no_unit_request_stop_s) / 60.0:.2f} / {no_unit_request_stop_s / 60.0:.2f} min"
        )

    if not t_parts:
        raise ValueError("No samples loaded from no-unit LFP/ME cache")
    no_unit_trace = lme_utils.TraceResult(
        np.concatenate(t_parts),
        np.concatenate(lfp_parts),
        np.concatenate(me_parts),
        float(cache_chunk.sample_rate_hz),
    )
    _no_unit_trace_settings = no_unit_trace_settings
    print(
        f"computed no-unit trace from cache chunks in {perf_counter() - no_unit_t0:.1f} s; "
        f"{no_unit_trace.t_video_s.size:,} samples at {no_unit_trace.sample_rate_hz:.1f} Hz"
    )
else:
    no_unit_trace = lme_utils.compute_lfp_magnitude_and_me(
        recording,
        no_unit_channels,
        start_s=no_unit_start_s,
        duration_s=no_unit_duration_s,
        chunk_seconds=NO_UNIT_CACHE_CHUNK_SECONDS,
        smooth_seconds=NO_UNIT_SMOOTH_SECONDS,
        target_rate_hz=no_unit_target_rate_hz,
        read_chunk_seconds=1.0,
        read_mode="auto",
        lfp_bandpass_hz=NO_UNIT_LFP_BANDPASS_HZ,
        use_cache=False,
    )
    _no_unit_trace_settings = no_unit_trace_settings
    print(
        f"computed no-unit trace from raw LF chunks in {perf_counter() - no_unit_t0:.1f} s; "
        f"{no_unit_trace.t_video_s.size:,} samples at {no_unit_trace.sample_rate_hz:.1f} Hz"
    )
if no_unit_trace.t_video_s.size == 0:
    raise ValueError("No samples in requested LFP/ME-only plot window")

no_unit_t0 = perf_counter()
no_unit_lfp_norm = lme_utils.percentile_minmax(
    no_unit_trace.lfp_trace,
    lower_percentile=1.0,
    upper_percentile=99.0,
)
no_unit_me_norm = lme_utils.percentile_minmax(
    no_unit_trace.motion_energy,
    lower_percentile=1.0,
    upper_percentile=99.0,
)
no_unit_r, no_unit_valid = lme_utils.pearsonr_valid(no_unit_trace.lfp_trace, no_unit_trace.motion_energy)
print(f"normalized/plotted no-unit traces in {perf_counter() - no_unit_t0:.1f} s")

no_unit_max_plot_points = 100_000
no_unit_plot_step = max(1, int(np.ceil(no_unit_trace.t_video_s.size / no_unit_max_plot_points)))
no_unit_plot_idx = np.arange(no_unit_trace.t_video_s.size)[::no_unit_plot_step]
no_unit_channel_label = lme_utils.channel_label(no_unit_channels)

fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
ax.plot(
    no_unit_trace.t_video_s[no_unit_plot_idx] / 60.0,
    no_unit_lfp_norm[no_unit_plot_idx],
    label="LFP envelope, 1-99% min-max",
    lw=0.9,
)
ax.plot(
    no_unit_trace.t_video_s[no_unit_plot_idx] / 60.0,
    no_unit_me_norm[no_unit_plot_idx],
    label="motion-energy envelope, 1-99% min-max",
    lw=0.9,
    color="tab:orange",
)
ax.set_title(
    f"LFP envelope and motion energy only, {no_unit_window_label}, "
    f"LF {no_unit_channel_label} (r={no_unit_r:+.3f}, n={no_unit_valid.sum():,})"
)
ax.set_xlim(no_unit_trace.t_video_s[0] / 60.0, no_unit_trace.t_video_s[-1] / 60.0)
ax.set_ylim(-1, 3.03)
ax.set_xlabel("video time (min)")
ax.set_ylabel("1-99% min-max normalized envelope")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


no-unit LFP/ME cache: /home/sam-reiter/bucket/ReiterU/Ants/physiology/20260715_antrec/ant1/lfp_me_cache_250hz_ch000-383.h5
bandpass filtering cached LFP and motion energy: 2-50 Hz; computing envelopes
processed cached chunk 1 through 1.00 / 132.31 min
bandpass filtering cached LFP and motion energy: 2-50 Hz; computing envelopes
processed cached chunk 2 through 2.00 / 132.31 min
bandpass filtering cached LFP and motion energy: 2-50 Hz; computing envelopes
processed cached chunk 3 through 3.00 / 132.31 min
bandpass filtering cached LFP and motion energy: 2-50 Hz; computing envelopes
processed cached chunk 4 through 4.00 / 132.31 min
bandpass filtering cached LFP and motion energy: 2-50 Hz; computing envelopes
processed cached chunk 5 through 5.00 / 132.31 min
bandpass filtering cached LFP and motion energy: 2-50 Hz; computing envelopes
processed cached chunk 6 through 6.00 / 132.31 min
bandpass filtering cached LFP and motion energy: 2-50 Hz; computing envelopes
processed cached chunk 7 

In [ ]:
# Load Kilosort spikes and map unit rates onto the LFP/ME timebase
KILOSORT_PATH = (
    EXPERIMENT_PATH
    / "20270709_ant_cb1_g0/20270709_ant_cb1_g0_imec0/"
    / "20270709_ant_cb1_g0_t0.imec0.ap.ks_0_81000000_allch_kilosort4"
)
KILOSORT_INCLUDE_LABELS = "good"     # use "all" to include all clusters
KILOSORT_EXCLUDE_LABELS = None
SPIKE_RATE_SMOOTH_SECONDS = 0.02

update_corr_trace_if_needed()
kilosort_spikes = load_kilosort_spikes(
    KILOSORT_PATH,
    recording=recording,
    include_labels=KILOSORT_INCLUDE_LABELS,
    exclude_labels=KILOSORT_EXCLUDE_LABELS,
)
spike_rate = map_spike_rates_to_timebase(
    kilosort_spikes,
    corr_trace.t_video_s,
    smoothing_sigma_s=SPIKE_RATE_SMOOTH_SECONDS,
)
if kilosort_spikes.sorted_binary_start_video_seconds is not None:
    spike_sorted_mask = corr_trace.t_video_s >= kilosort_spikes.sorted_binary_start_video_seconds
else:
    spike_sorted_mask = np.ones(corr_trace.t_video_s.shape, dtype=bool)
if kilosort_spikes.sorted_binary_stop_video_seconds is not None:
    spike_sorted_mask &= corr_trace.t_video_s < kilosort_spikes.sorted_binary_stop_video_seconds

spike_lfp_r = correlate_unit_rates_to_trace(
    spike_rate,
    corr_trace.lfp_trace,
    valid_mask=spike_sorted_mask,
)
spike_me_r = correlate_unit_rates_to_trace(
    spike_rate,
    corr_trace.motion_energy,
    valid_mask=spike_sorted_mask,
)
print(
    f"Loaded {len(kilosort_spikes.unit_ids):,} Kilosort units from {kilosort_spikes.path}; "
    f"{len(kilosort_spikes.spike_times_samples):,} spikes, "
    f"{kilosort_spikes.sample_rate_hz:g} Hz"
)
if kilosort_spikes.sorted_binary_start_video_seconds is not None:
    print(
        "Kilosort sorted window on video axis: "
        f"{kilosort_spikes.sorted_binary_start_video_seconds:.3f}-"
        f"{kilosort_spikes.sorted_binary_stop_video_seconds:.3f} s; "
        f"overlap samples on corr_trace={spike_sorted_mask.sum():,}"
    )
print(
    f"Mapped spike rates: {spike_rate.spike_rate_hz.shape[0]:,} unit(s) x "
    f"{spike_rate.spike_rate_hz.shape[1]:,} samples at {corr_trace.sample_rate_hz:.1f} Hz; "
    f"smoothing sigma={SPIKE_RATE_SMOOTH_SECONDS:g}s"
)

rank_metric = np.maximum(
    np.nan_to_num(np.abs(spike_lfp_r), nan=-np.inf),
    np.nan_to_num(np.abs(spike_me_r), nan=-np.inf),
)
ranked_units = np.argsort(rank_metric)[::-1]
print("Top unit-rate correlations:")
for rank_idx in ranked_units[:10]:
    unit_id = int(spike_rate.unit_ids[rank_idx])
    label = spike_rate.unit_labels[rank_idx] or "unlabeled"
    mean_rate = float(np.nanmean(spike_rate.spike_rate_hz[rank_idx, spike_sorted_mask]))
    print(
        f"  unit {unit_id:>4d} ({label:>4s}): "
        f"r_lfp={spike_lfp_r[rank_idx]:+.3f}, "
        f"r_me={spike_me_r[rank_idx]:+.3f}, "
        f"mean_rate={mean_rate:.2f} Hz"
    )


In [ ]:
# Screen for unit activity that leads later LFP/ME increases
RAMP_ANALYSIS_RATE_HZ = 20.0
RAMP_MAX_LEAD_SECONDS = 30.0
RAMP_LAG_STEP_SECONDS = 1.0
RAMP_MIN_LEAD_SECONDS = 1.0
RAMP_FUTURE_LEAD_SECONDS = 2.0
RAMP_FUTURE_WINDOW_SECONDS = 10.0
RAMP_TRACE_SMOOTH_SECONDS = 2.0
RAMP_ZERO_LAG_PENALTY = 1.0
RAMP_TOP_UNITS = 12

update_corr_trace_if_needed()
ramp_step = max(1, int(round(corr_trace.sample_rate_hz / RAMP_ANALYSIS_RATE_HZ)))
ramp_rate_hz = corr_trace.sample_rate_hz / ramp_step
ramp_spike_rate = decimate_spike_rate_result(spike_rate, ramp_step)
ramp_lfp_trace = corr_trace.lfp_trace[::ramp_step]
ramp_me_trace = corr_trace.motion_energy[::ramp_step]
ramp_valid_mask = spike_sorted_mask[::ramp_step]

lfp_lead = unit_trace_lag_correlations(
    ramp_spike_rate,
    ramp_lfp_trace,
    sample_rate_hz=ramp_rate_hz,
    max_lag_seconds=RAMP_MAX_LEAD_SECONDS,
    lag_step_seconds=RAMP_LAG_STEP_SECONDS,
    valid_mask=ramp_valid_mask,
    min_lead_seconds=RAMP_MIN_LEAD_SECONDS,
)
me_lead = unit_trace_lag_correlations(
    ramp_spike_rate,
    ramp_me_trace,
    sample_rate_hz=ramp_rate_hz,
    max_lag_seconds=RAMP_MAX_LEAD_SECONDS,
    lag_step_seconds=RAMP_LAG_STEP_SECONDS,
    valid_mask=ramp_valid_mask,
    min_lead_seconds=RAMP_MIN_LEAD_SECONDS,
)
lfp_future_change, lfp_future_valid = future_trace_change(
    ramp_lfp_trace,
    sample_rate_hz=ramp_rate_hz,
    lead_seconds=RAMP_FUTURE_LEAD_SECONDS,
    change_window_seconds=RAMP_FUTURE_WINDOW_SECONDS,
    smooth_seconds=RAMP_TRACE_SMOOTH_SECONDS,
    valid_mask=ramp_valid_mask,
)
me_future_change, me_future_valid = future_trace_change(
    ramp_me_trace,
    sample_rate_hz=ramp_rate_hz,
    lead_seconds=RAMP_FUTURE_LEAD_SECONDS,
    change_window_seconds=RAMP_FUTURE_WINDOW_SECONDS,
    smooth_seconds=RAMP_TRACE_SMOOTH_SECONDS,
    valid_mask=ramp_valid_mask,
)
lfp_future_r = correlate_unit_rates_to_trace(
    ramp_spike_rate,
    lfp_future_change,
    valid_mask=ramp_valid_mask & lfp_future_valid,
)
me_future_r = correlate_unit_rates_to_trace(
    ramp_spike_rate,
    me_future_change,
    valid_mask=ramp_valid_mask & me_future_valid,
)

lead_lfp_r = np.nan_to_num(lfp_lead.best_lead_r, nan=-np.inf)
lead_me_r = np.nan_to_num(me_lead.best_lead_r, nan=-np.inf)
future_lfp_r = np.nan_to_num(lfp_future_r, nan=-np.inf)
future_me_r = np.nan_to_num(me_future_r, nan=-np.inf)
zero_metric = np.maximum(
    np.nan_to_num(np.abs(spike_lfp_r), nan=0.0),
    np.nan_to_num(np.abs(spike_me_r), nan=0.0),
)
lead_metric = np.maximum(lead_lfp_r, lead_me_r)
future_metric = np.maximum(future_lfp_r, future_me_r)
ramp_score = np.maximum(lead_metric, future_metric) - RAMP_ZERO_LAG_PENALTY * zero_metric
ramp_ranked_units = np.argsort(ramp_score)[::-1]

print(
    f"Ramping/lead screen at {ramp_rate_hz:.1f} Hz: positive lead means unit activity "
    "comes before LFP/ME. Future-change score asks whether current firing predicts "
    f"a trace rise {RAMP_FUTURE_LEAD_SECONDS:g}-{RAMP_FUTURE_LEAD_SECONDS + RAMP_FUTURE_WINDOW_SECONDS:g}s later."
)
print("Top ramp/lead candidates:")
for rank_idx in ramp_ranked_units[:RAMP_TOP_UNITS]:
    unit_id = int(spike_rate.unit_ids[rank_idx])
    label = spike_rate.unit_labels[rank_idx] or "unlabeled"
    if lead_lfp_r[rank_idx] >= lead_me_r[rank_idx]:
        lead_source = "lfp"
        lead_s = lfp_lead.best_lead_s[rank_idx]
        lead_r = lfp_lead.best_lead_r[rank_idx]
    else:
        lead_source = "me"
        lead_s = me_lead.best_lead_s[rank_idx]
        lead_r = me_lead.best_lead_r[rank_idx]
    if future_lfp_r[rank_idx] >= future_me_r[rank_idx]:
        future_source = "lfp"
        future_r = lfp_future_r[rank_idx]
    else:
        future_source = "me"
        future_r = me_future_r[rank_idx]
    print(
        f"  unit {unit_id:>4d} ({label:>4s}): "
        f"score={ramp_score[rank_idx]:+.3f}, "
        f"best_lead={lead_r:+.3f} {lead_source} at {lead_s:.1f}s, "
        f"future_r={future_r:+.3f} {future_source}, "
        f"zero_lfp={spike_lfp_r[rank_idx]:+.3f}, zero_me={spike_me_r[rank_idx]:+.3f}"
    )


In [ ]:
# Plot unit spike rates aligned to LFP magnitude and motion energy
SPIKE_RATE_PLOT_ORDER = "low_me_rate"      # "low_me_rate", "ramp", or "correlation"
SPIKE_RATE_LOW_ME_PERCENTILE = 10.0
SPIKE_RATE_PLOT_SMOOTH_SECONDS = 1.0
SPIKE_RATE_MAX_PLOT_UNITS = 40
SPIKE_RATE_MAX_PLOT_POINTS = 100_000

update_corr_trace_if_needed()
spike_plot_start_min = 0.0
spike_plot_duration_min = 45.0
spike_plot_start_s = spike_plot_start_min * 60.0
spike_plot_stop_s = spike_plot_start_s + spike_plot_duration_min * 60.0
spike_plot_mask = (
    (corr_trace.t_video_s >= spike_plot_start_s)
    & (corr_trace.t_video_s < spike_plot_stop_s)
    & spike_sorted_mask
)
spike_plot_full_idx = np.flatnonzero(spike_plot_mask)
if spike_plot_full_idx.size == 0:
    raise ValueError("No Kilosort/LFP/ME overlap in requested spike-rate plot window")

if SPIKE_RATE_PLOT_ORDER == "low_me_rate":
    low_me_threshold = float(
        np.nanpercentile(corr_trace.motion_energy[spike_plot_full_idx], SPIKE_RATE_LOW_ME_PERCENTILE)
    )
    low_me_idx = spike_plot_full_idx[corr_trace.motion_energy[spike_plot_full_idx] <= low_me_threshold]
    if low_me_idx.size == 0:
        low_me_idx = spike_plot_full_idx
    low_me_mean_rate = np.nanmean(spike_rate.spike_rate_hz[:, low_me_idx], axis=1)
    unit_order_source = np.argsort(np.nan_to_num(low_me_mean_rate, nan=-np.inf))[::-1]
    order_detail = (
        f"low ME p{SPIKE_RATE_LOW_ME_PERCENTILE:g}, "
        f"n={low_me_idx.size:,}, threshold={low_me_threshold:.3g}"
    )
elif SPIKE_RATE_PLOT_ORDER == "ramp" and "ramp_ranked_units" in globals():
    unit_order_source = ramp_ranked_units
    order_detail = "ramp"
else:
    unit_order_source = ranked_units
    order_detail = "correlation"

spike_plot_step = max(1, int(np.ceil(spike_plot_full_idx.size / SPIKE_RATE_MAX_PLOT_POINTS)))
spike_plot_idx = spike_plot_full_idx[::spike_plot_step]
unit_plot_order = unit_order_source[: min(SPIKE_RATE_MAX_PLOT_UNITS, unit_order_source.size)]
rate_image = spike_rate.spike_rate_hz[unit_plot_order][:, spike_plot_idx]
if SPIKE_RATE_PLOT_SMOOTH_SECONDS > 0 and spike_plot_idx.size > 1:
    plot_rate_hz = 1.0 / float(np.nanmedian(np.diff(corr_trace.t_video_s[spike_plot_idx])))
    rate_image = gaussian_smooth_rows(rate_image, SPIKE_RATE_PLOT_SMOOTH_SECONDS * plot_rate_hz)
rate_image_z = np.vstack([zscore(row) for row in rate_image]).astype(np.float32, copy=False)

fig, axes = plt.subplots(
    3,
    1,
    figsize=(13, 9),
    sharex=True,
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1.0, 1.0, 2.4]},
)
axes[0].plot(
    corr_trace.t_video_s[spike_plot_idx] / 60.0,
    zscore(corr_trace.lfp_trace[spike_plot_idx]),
    lw=0.9,
    label=corr_lfp_label,
)
axes[0].legend(loc="upper right")
axes[0].set_ylabel("LFP z")
axes[0].grid(alpha=0.25)

axes[1].plot(
    corr_trace.t_video_s[spike_plot_idx] / 60.0,
    zscore(corr_trace.motion_energy[spike_plot_idx]),
    lw=0.9,
    color="tab:orange",
    label=corr_me_label,
)
axes[1].legend(loc="upper right")
axes[1].set_ylabel("ME z")
axes[1].grid(alpha=0.25)

vlim = np.nanpercentile(np.abs(rate_image_z), 98) if np.isfinite(rate_image_z).any() else 1.0
im = axes[2].imshow(
    rate_image_z,
    aspect="auto",
    interpolation="nearest",
    cmap="RdBu_r",
    vmin=-vlim,
    vmax=vlim,
    extent=[
        corr_trace.t_video_s[spike_plot_idx[0]] / 60.0,
        corr_trace.t_video_s[spike_plot_idx[-1]] / 60.0,
        len(unit_plot_order) - 0.5,
        -0.5,
    ],
)
axes[2].set_yticks(np.arange(len(unit_plot_order)))
axes[2].set_yticklabels([str(int(spike_rate.unit_ids[idx])) for idx in unit_plot_order])
axes[2].set_ylabel("unit id")
axes[2].set_xlabel("video time (min)")
axes[2].set_title(
    "Kilosort unit spike rates, "
    f"order={order_detail}, plot smoothing={SPIKE_RATE_PLOT_SMOOTH_SECONDS:g}s"
)
fig.colorbar(im, ax=axes[2], label="spike-rate z")
plt.show()


In [ ]:
# Plot correlation result
update_corr_trace_if_needed()
plot_start_min = 0.0
plot_duration_min = 45.0
plot_start_s = plot_start_min * 60.0
plot_stop_s = plot_start_s + plot_duration_min * 60.0
plot_mask = (corr_trace.t_video_s >= plot_start_s) & (corr_trace.t_video_s < plot_stop_s)
plot_t_s = corr_trace.t_video_s[plot_mask]
if plot_t_s.size == 0:
    raise ValueError("No samples in requested plot window")

plot_step = max(1, int(np.ceil(plot_t_s.size / 100_000)))
plot_idx = np.flatnonzero(plot_mask)[::plot_step]
plot_valid_idx = np.flatnonzero(corr_valid & plot_mask)
scatter_step = max(1, int(np.ceil(plot_valid_idx.size / 75_000)))
corr_valid_idx = plot_valid_idx[::scatter_step]

fig, axes = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True)
axes[0].plot(
    corr_trace.t_video_s[plot_idx] / 60.0,
    zscore(corr_trace.lfp_trace[plot_mask])[::plot_step],
    label=corr_lfp_label,
    lw=0.9,
)
axes[0].plot(
    corr_trace.t_video_s[plot_idx] / 60.0,
    zscore(corr_trace.motion_energy[plot_mask])[::plot_step],
    label=corr_me_label,
    lw=0.9,
)
axes[0].set_title(
    f"{corr_trace.sample_rate_hz:.1f} Hz LFP-sampled signals, first {plot_duration_min:g} min, "
    f"LF channels {corr_channels[0]}-{corr_channels[-1]} (r={corr_r:.3f})"
)
axes[0].set_xlim(plot_start_min, plot_start_min + plot_duration_min)
axes[0].set_xlabel("video time (min)")
axes[0].set_ylabel("z-score")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].scatter(
    corr_trace.motion_energy[corr_valid_idx],
    corr_trace.lfp_trace[corr_valid_idx],
    s=3,
    alpha=0.15,
)
axes[1].set_xlabel("motion-energy envelope at LFP sample times")
axes[1].set_ylabel("mean LFP envelope (uV)")
axes[1].set_title("Envelope samples, downsampled for visualization")
axes[1].grid(alpha=0.25)
plt.show()


In [ ]:
# Whole-dataset channel-average correlation with block-local filter parameters
FULL_CORR_CHANNEL_START = 150
FULL_CORR_CHANNEL_STOP = 151
FULL_CORR_CHANNEL_STEP = 1
FULL_CORR_CHUNK_SECONDS = 3600.0
FULL_CORR_REDUCTION = "mean"     # "mean" or "mean_abs"
FULL_CORR_LFP_HIGHPASS_HZ = None
FULL_CORR_LFP_LOWPASS_HZ = 5.0
FULL_CORR_ME_HIGHPASS_HZ = None
FULL_CORR_ME_LOWPASS_HZ = 5.0
FULL_CORR_FILTER_ORDER = 3
FULL_CORR_MAX_PLOT_POINTS = 100_000
FULL_CORR_MAX_SCATTER_POINTS = 75_000
TARGET_RATE_HZ = 250.0
LFP_READ_CHUNK_SECONDS = 1.0
LFP_READ_MODE = "auto"

full_channels = channel_range(FULL_CORR_CHANNEL_START, FULL_CORR_CHANNEL_STOP, FULL_CORR_CHANNEL_STEP)
importlib.reload(lme_utils)
full_t, full_lfp_mean, full_sample_rate_hz = lme_utils.load_channel_average_lfp(
    recording,
    full_channels,
    chunk_seconds=FULL_CORR_CHUNK_SECONDS,
    target_rate_hz=TARGET_RATE_HZ,
    read_chunk_seconds=LFP_READ_CHUNK_SECONDS,
    read_mode=LFP_READ_MODE,
    reduction=FULL_CORR_REDUCTION,
)
full_me_cache_path = lme_utils.find_lfp_motion_energy_cache(
    recording,
    full_channels,
    target_rate_hz=TARGET_RATE_HZ,
)
if full_me_cache_path is not None:
    full_me_cache = lme_utils.load_lfp_motion_energy_cache(
        full_me_cache_path,
        channels=full_channels,
        load_lfp=False,
    )
    full_me = full_me_cache.motion_energy
else:
    full_me = recording.sample_motion_energy(full_t)
full_lfp_filtered = butter_filter_1d(
    full_lfp_mean,
    sample_rate_hz=full_sample_rate_hz,
    highpass_hz=FULL_CORR_LFP_HIGHPASS_HZ,
    lowpass_hz=FULL_CORR_LFP_LOWPASS_HZ,
    order=FULL_CORR_FILTER_ORDER,
)
full_me_filtered = butter_filter_1d(
    full_me,
    sample_rate_hz=full_sample_rate_hz,
    highpass_hz=FULL_CORR_ME_HIGHPASS_HZ,
    lowpass_hz=FULL_CORR_ME_LOWPASS_HZ,
    order=FULL_CORR_FILTER_ORDER,
)
full_lfp_envelope = amplitude_envelope_1d(full_lfp_filtered)
full_me_envelope = amplitude_envelope_1d(full_me_filtered)

full_r, full_valid = pearsonr_valid(full_lfp_envelope, full_me_envelope)
print(
    f"Whole-dataset filtered-envelope Pearson r = {full_r:+.4f} over {full_valid.sum():,} samples "
    f"at {full_sample_rate_hz:.1f} Hz; "
    f"channels {full_channels[0]}-{full_channels[-1]}, reduction={FULL_CORR_REDUCTION}"
)

full_plot_step = max(1, int(np.ceil(len(full_t) / FULL_CORR_MAX_PLOT_POINTS)))
full_scatter_step = max(1, int(np.ceil(full_valid.sum() / FULL_CORR_MAX_SCATTER_POINTS)))
full_scatter_idx = np.flatnonzero(full_valid)[::full_scatter_step]

fig, axes = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True)
axes[0].plot(
    full_t[::full_plot_step] / 60.0,
    zscore(full_lfp_envelope)[::full_plot_step],
    label="filtered LFP envelope",
    lw=0.9,
)
axes[0].plot(
    full_t[::full_plot_step] / 60.0,
    zscore(full_me_envelope)[::full_plot_step],
    label="filtered ME envelope",
    lw=0.9,
)
axes[0].set_title(
    f"Whole-dataset filtered-envelope correlation at {full_sample_rate_hz:.1f} Hz, "
    f"LF channels {full_channels[0]}-{full_channels[-1]} (r={full_r:+.3f})"
)
axes[0].set_xlabel("video time (min)")
axes[0].set_ylabel("z-score")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].scatter(
    full_me_envelope[full_scatter_idx],
    full_lfp_envelope[full_scatter_idx],
    s=4,
    alpha=0.18,
)
axes[1].set_title("Envelope samples, downsampled for display")
axes[1].set_xlabel("filtered motion-energy envelope")
axes[1].set_ylabel("filtered channel-average LFP envelope (uV)")
axes[1].grid(alpha=0.25)
plt.show()


In [ ]:
# Diagnostic block: compare low-pass choices for ME and LFP envelopes
DIAG_LOWPASS_HZ = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
DIAG_TRACE_LOWPASS_HZ = 1.0
DIAG_LAG_SECONDS = 30.0
DIAG_LAG_RATE_HZ = 20.0

diagnostic_rows = []
for cutoff_hz in DIAG_LOWPASS_HZ:
    lfp_f = butter_filter_1d(
        corr_trace.lfp_trace,
        sample_rate_hz=corr_trace.sample_rate_hz,
        lowpass_hz=cutoff_hz,
        order=3,
    )
    me_f = butter_filter_1d(
        corr_trace.motion_energy,
        sample_rate_hz=corr_trace.sample_rate_hz,
        lowpass_hz=cutoff_hz,
        order=3,
    )
    r0, valid_f = pearsonr_valid(lfp_f, me_f)
    me_lag, lag_rate_hz = decimate_for_lag(zscore(me_f), corr_trace.sample_rate_hz, DIAG_LAG_RATE_HZ)
    lfp_lag, _lag_rate_hz = decimate_for_lag(zscore(lfp_f), corr_trace.sample_rate_hz, DIAG_LAG_RATE_HZ)
    lags_s, lag_r = lagged_pearson(
        me_lag,
        lfp_lag,
        sample_rate_hz=lag_rate_hz,
        max_lag_seconds=DIAG_LAG_SECONDS,
    )
    best_idx = int(np.nanargmax(np.abs(lag_r))) if np.isfinite(lag_r).any() else 0
    diagnostic_rows.append(
        {
            "cutoff_hz": float(cutoff_hz),
            "r_zero_lag": float(r0),
            "best_lag_s": float(lags_s[best_idx]),
            "best_lag_r": float(lag_r[best_idx]),
            "lfp_filtered": lfp_f,
            "me_filtered": me_f,
            "lags_s": lags_s,
            "lag_r": lag_r,
        }
    )

print("Low-pass diagnostic:")
for row in diagnostic_rows:
    print(
        f"  <= {row['cutoff_hz']:>4g} Hz: "
        f"r0={row['r_zero_lag']:+.3f}, "
        f"best |r|={row['best_lag_r']:+.3f} at lag={row['best_lag_s']:+.2f}s"
    )

trace_row = min(diagnostic_rows, key=lambda row: abs(row["cutoff_hz"] - float(DIAG_TRACE_LOWPASS_HZ)))
plot_step = max(1, int(np.ceil(len(corr_trace.t_video_s) / 100_000)))
valid_trace = np.isfinite(trace_row["lfp_filtered"]) & np.isfinite(trace_row["me_filtered"])
scatter_step = max(1, int(np.ceil(valid_trace.sum() / 75_000)))
scatter_idx = np.flatnonzero(valid_trace)[::scatter_step]

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
axes[0, 0].plot(
    corr_trace.t_video_s[::plot_step] / 60.0,
    zscore(trace_row["lfp_filtered"])[::plot_step],
    label="low-pass LFP envelope",
    lw=0.9,
)
axes[0, 0].plot(
    corr_trace.t_video_s[::plot_step] / 60.0,
    zscore(trace_row["me_filtered"])[::plot_step],
    label="low-pass ME envelope",
    lw=0.9,
)
axes[0, 0].set_title(f"Low-pass <= {trace_row['cutoff_hz']:g} Hz")
axes[0, 0].set_xlabel("video time (min)")
axes[0, 0].set_ylabel("z-score")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

axes[0, 1].scatter(
    trace_row["me_filtered"][scatter_idx],
    trace_row["lfp_filtered"][scatter_idx],
    s=4,
    alpha=0.18,
)
axes[0, 1].set_title(f"Scatter, r0={trace_row['r_zero_lag']:+.3f}")
axes[0, 1].set_xlabel("low-pass ME envelope")
axes[0, 1].set_ylabel("low-pass LFP envelope (uV)")
axes[0, 1].grid(alpha=0.25)

cutoffs = np.asarray([row["cutoff_hz"] for row in diagnostic_rows], dtype=float)
r0 = np.asarray([row["r_zero_lag"] for row in diagnostic_rows], dtype=float)
best_r = np.asarray([row["best_lag_r"] for row in diagnostic_rows], dtype=float)
axes[1, 0].semilogx(cutoffs, r0, marker="o", label="zero lag")
axes[1, 0].semilogx(cutoffs, best_r, marker="o", label=f"best lag within +/-{DIAG_LAG_SECONDS:g}s")
axes[1, 0].axhline(0, color="0.5", lw=0.8)
axes[1, 0].set_title("Correlation vs low-pass cutoff")
axes[1, 0].set_xlabel("low-pass cutoff (Hz)")
axes[1, 0].set_ylabel("Pearson r")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25, which="both")

axes[1, 1].plot(trace_row["lags_s"], trace_row["lag_r"], color="tab:purple")
axes[1, 1].axvline(0, color="0.5", lw=0.8)
axes[1, 1].axhline(0, color="0.5", lw=0.8)
axes[1, 1].set_title(f"Lag correlation, <= {trace_row['cutoff_hz']:g} Hz")
axes[1, 1].set_xlabel("lag (s), positive = ME leads LFP")
axes[1, 1].set_ylabel("Pearson r")
axes[1, 1].grid(alpha=0.25)
plt.show()
